# imports

In [ ]:
import numpy as numpy
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# load data

In [ ]:
# src : https://www.kaggle.com/datasets/jahaidulislam/car-specification-dataset-1945-2020?select=Car+Dataset+1945-2020.csv
df = pd.read_csv("res/car-dataset-1945-2020.csv")
df['city_fuel_per_100km_l'].isnull().sum()

#Car with the highest hp
df['engine_hp'].idxmax()
df.loc[df['engine_hp'].idxmax()]



In [ ]:
column = 'CO2_emissions_g/km'

plt.figure(figsize=(10, 6))
sns.histplot(df[column].dropna(), kde=True)
plt.title(f'Distribution of {column}')
plt.xlabel(column)
plt.ylabel('Frequency')
plt.show()

#type(df['number_of_seats'].head(1)[0])
#df[column].between(0, 500).sum()


In [ ]:
df.info()

# preprocessing

In [ ]:
# Remove unnecessary columns and rename selected columns

columns_to_keep = [
    # Main
    "Make",
    "Modle",
    "Year_from",
    "Year_to",
    "Body_type",
    "car_class",
    "number_of_seats",

    # Engine
    "engine_type",
    "capacity_cm3",
    "engine_hp",
    "maximum_torque_n_m",
    "number_of_cylinders",
    "cylinder_layout",

    # Weight
    "curb_weight_kg",

    # Consumption
    "mixed_fuel_consumption_per_100_km_l",
    "city_fuel_per_100km_l",
    "highway_fuel_per_100km_l",
    "CO2_emissions_g/km",
    "emission_standards",
    "range_km",
    "fuel_grade",

    # Performance
    "acceleration_0_100_km/h_s",
    "max_speed_km_per_h",
]

df = df.loc[:, columns_to_keep].rename(
    columns={
        "Make": "manufacturer",
        "Modle": "model",
        "Year_from": "year_from",
        "Year_to": "year_to",
        "Body_type": "body_type",
        "car_class": "car_class",
        "CO2_emissions_g/km": "co2_emissions_g_per_km",
        "acceleration_0_100_km/h_s": "acceleration_0_100_km_h_s",
    }
)
df

In [ ]:
df.describe()

In [ ]:
# Numeric cleaning

# 1. "range_km" stores "city|highway" with thousands separators, e.g. "450|1,000"
range_parts = df["range_km"].astype(str).str.split("|", expand=True)
df["range_city_km"] = pd.to_numeric(
    range_parts[0].str.replace(",", "", regex=False), errors="coerce"
)
df["range_highway_km"] = pd.to_numeric(
    range_parts[1].str.replace(",", "", regex=False), errors="coerce"
)
df = df.drop(columns=["range_km"])

# 2. some columns use "," as a separator, e.g. "157,4" == 157.4, we replace by point only
decimal_comma_cols = [
    "capacity_cm3",
    "maximum_torque_n_m",
    "curb_weight_kg",
    "max_speed_km_per_h",
]
for col in decimal_comma_cols:
    df[col] = pd.to_numeric(
        df[col].astype(str).str.replace(",", ".", regex=False),
        errors="coerce",
    )

# 3. "number_of_seats" stores multi-config as a list, e.g. "5, 7" (5 or 7 seater), we take the max
seats = df["number_of_seats"].astype(str).str.split(",", expand=True)
df["number_of_seats"] = seats.apply(pd.to_numeric, errors="coerce").max(axis=1)

# Other purely-numeric columns
plain_numeric_cols = [
    "year_from",
    "year_to",
    "engine_hp",
    "number_of_cylinders",
    "mixed_fuel_consumption_per_100_km_l",
    "city_fuel_per_100km_l",
    "highway_fuel_per_100km_l",
    "co2_emissions_g_per_km",
    "acceleration_0_100_km_h_s",
]
df[plain_numeric_cols] = df[plain_numeric_cols].apply(pd.to_numeric, errors="coerce")

df.info()

In [ ]:
df.head(50)

In [ ]:
# normalize categorical text columns (case + variant spellings)
df["engine_type"] = (
    df["engine_type"]
    .str.lower()
    .replace({"petrol": "gasoline", "diesel fuel": "diesel"})
)

df["cylinder_layout"] = (
    df["cylinder_layout"].str.lower().replace({"-": pd.NA})
)

df["body_type"] = df["body_type"].str.lower()

# emission_standards: unify Roman ("EURO IV") and Arabic ("Euro 4") variants
euro_map = {
    "EURO I": "EURO 1", "EURO II": "EURO 2", "EURO III": "EURO 3",
    "EURO IV": "EURO 4", "EURO V": "EURO 5", "EURO VI": "EURO 6",
    "Euro 2": "EURO 2", "Euro 3": "EURO 3", "Euro 4": "EURO 4",
    "Euro 5": "EURO 5", "Euro 6": "EURO 6",
}
df["emission_standards"] = df["emission_standards"].replace(euro_map)

# fuel_grade: take first listed value (most rows have a single octane like "95")
df["fuel_grade"] = (
    df["fuel_grade"].str.split(",").str[0].str.strip().str.lower()
)

df[["engine_type", "cylinder_layout", "body_type", "emission_standards", "fuel_grade"]].nunique()

In [ ]:
# Check missing values
df.dropna(how="any")

# efficiency ratios

In [ ]:
df["hp_per_fuel"] = df["engine_hp"] / df["mixed_fuel_consumption_per_100_km_l"]
df["co2_per_hp"] = df["co2_emissions_g_per_km"] / df["engine_hp"]
df["fuel_per_weight"] = df["mixed_fuel_consumption_per_100_km_l"] / df["curb_weight_kg"]
df['power_density'] = df['engine_hp'] / df['capacity_cm3']
df['weight_power_ratio'] = df['engine_hp'] / df['curb_weight_kg']
df['city_highway_ratio'] = df['city_fuel_per_100km_l'] / df['highway_fuel_per_100km_l']
df['co2_per_seat'] = df['co2_emissions_g_per_km'] / df['number_of_seats']

emission_factor_g_per_l = df['engine_type'].map({'gasoline': 2310, 'diesel': 2650})
df['estimated_co2_g_per_km'] = (
    df['mixed_fuel_consumption_per_100_km_l'] / 100 * emission_factor_g_per_l
)

# HP per cylinder power extracted per combustion chamber.
df['hp_per_cylinder'] = df['engine_hp'] / df['number_of_cylinders']

df[[
    "manufacturer", "model", "year_from",
    "engine_hp", "mixed_fuel_consumption_per_100_km_l",
    "curb_weight_kg", "co2_emissions_g_per_km",
    "hp_per_fuel", "co2_per_hp", "fuel_per_weight",
    "power_density", "weight_power_ratio", "city_highway_ratio", "co2_per_seat",
    "estimated_co2_g_per_km", "hp_per_cylinder",
]].head()

# test

# plots

In [ ]:
mask_est = df['year_from'].between(1970, 2020) & df['estimated_co2_g_per_km'].notna()
yearly_est = (df.loc[mask_est].groupby('year_from')['estimated_co2_g_per_km']
              .agg(median='median', q25=lambda s: s.quantile(0.25),
                   q75=lambda s: s.quantile(0.75), n='count').reset_index())
yearly_est = yearly_est[yearly_est['n'] >= 20]

mask_dir = df['year_from'].between(1970, 2020) & df['co2_emissions_g_per_km'].notna()
yearly_dir = (df.loc[mask_dir].groupby('year_from')['co2_emissions_g_per_km']
              .agg(median='median', n='count').reset_index())
yearly_dir = yearly_dir[yearly_dir['n'] >= 20]

fig, ax = plt.subplots(figsize=(11, 6))
ax.fill_between(yearly_est['year_from'], yearly_est['q25'], yearly_est['q75'],
                alpha=0.2, color='steelblue', label='Estimated CO2 IQR')
ax.plot(yearly_est['year_from'], yearly_est['median'], linewidth=2.2, color='steelblue',
        label='Estimated CO2  median (from fuel)')
ax.plot(yearly_dir['year_from'], yearly_dir['median'], linewidth=2.2, color='crimson',
        marker='o', markersize=4, label='Direct CO2 median (measured)')
for name, year in {'EURO 1':1992,'EURO 3':2000,'EURO 4':2005,'EURO 6':2014}.items():
    ax.axvline(year, color='grey', linestyle='--', alpha=0.5, linewidth=1)
    ax.text(year+0.2, yearly_est['q75'].max()*0.98, name, rotation=90, fontsize=8, color='grey', va='top')
ax.set_xlabel('Year'); ax.set_ylabel('CO2 emissions (g/km)')
ax.set_title('CO2 emissions over time fuel-based estimate vs direct measurements, 1970-2020')
ax.legend(loc='upper right'); ax.grid(True, alpha=0.3); plt.tight_layout()
plt.show()

overlap = yearly_est.merge(yearly_dir, on='year_from', suffixes=('_est','_dir'))
overlap['ratio'] = overlap['median_est']/overlap['median_dir']
print(f"Median ratio (est/dir) across overlap: {overlap['ratio'].median():.3f}")

In [ ]:
#power density over time
mask = df['year_from'].between(1970, 2020) & df['power_density'].notna()
yearly_power_density = (df.loc[mask].groupby('year_from')['power_density']
                         .agg(median='median', q25=lambda s: s.quantile(0.25),
                              q75=lambda s: s.quantile(0.75), n='count').reset_index())
yearly_power_density = yearly_power_density[yearly_power_density['n'] >= 20]
fig, ax = plt.subplots(figsize=(11, 6))
ax.fill_between(yearly_power_density['year_from'], yearly_power_density['q25'], yearly_power_density['q75'],
                alpha=0.2, color='steelblue', label='Power Density IQR')
ax.plot(yearly_power_density['year_from'], yearly_power_density['median'], linewidth=2.2, color='steelblue',
        label='Power Density median (hp/cm3)')
ax.set_xlabel('Year'); ax.set_ylabel('Power Density (hp/cm3)')
ax.set_title('Engine Power Density over time, 1970-2020')
ax.legend(loc='upper left'); ax.grid(True, alpha=0.3); plt.tight_layout()
plt.show()

## Sample composition

In [ ]:
hp_bins = [0, 100, 150, 250, 400, 1e6]
hp_labels = [
    '< 100 HP ',
    '100-150 HP ',
    '150-250 HP ',
    '250-400 HP (high performance)',
    '> 400 HP (supercar / hypercar)',
]
mask = df['engine_hp'].notna() & df['year_from'].between(1960, 2020)
plot_df = df.loc[mask, ['year_from', 'engine_hp']].copy()
plot_df['decade'] = (plot_df['year_from'] // 10 * 10).astype(int).astype(str) + 's'
plot_df['hp_tier'] = pd.cut(plot_df['engine_hp'], bins=hp_bins, labels=hp_labels, right=False)

counts = (plot_df.groupby(['decade', 'hp_tier'], observed=True).size()
          .unstack(fill_value=0).reindex(columns=hp_labels, fill_value=0))
share = counts.div(counts.sum(axis=1), axis=0) * 100
decade_n = counts.sum(axis=1)

fig, ax = plt.subplots(figsize=(12, 6))

colors = ['#2c7a3f', '#7fbf7f', '#f4d35e', '#e8893f', '#c0392b']

share.plot(
    kind='bar',
    stacked=True,
    ax=ax,
    color=colors,
    width=0.85
)

ax.set_xlabel('Decade')
ax.set_ylabel('Share of dataset (%)')

ax.set_title(
    'Car-tier composition by decade'
)

ax.set_ylim(0, 105)

ax.legend(
    loc='upper left',
    bbox_to_anchor=(1.02, 1),
    title='HP tier'
)

ax.grid(True, alpha=0.3, axis='y')
plt.xticks(rotation=0)


plt.tight_layout()
plt.show()

print("Share of >250 HP cars per decade :")
print(share[['250-400 HP (high performance)', '> 400 HP (supercar / hypercar)']]
      .sum(axis=1).round(1).to_string())
print("\nNumber of cars per decade:")
print(decade_n.to_string())


In [ ]:
engine_bins = [0, 1000, 1500, 2000, 3000, 1e6]
engine_labels = [
    '< 1.0L',
    '1.0-1.5L',
    '1.5-2.0L',
    '2.0-3.0L',
    '> 3.0L',
]

mask = df['capacity_cm3'].notna() & df['year_from'].between(1960, 2020)
plot_df = df.loc[mask, ['year_from', 'capacity_cm3']].copy()
plot_df['decade'] = (plot_df['year_from'] // 10 * 10).astype(int).astype(str) + 's'
plot_df['engine_size_tier'] = pd.cut(
    plot_df['capacity_cm3'],
    bins=engine_bins,
    labels=engine_labels,
    right=False
)

counts = (
    plot_df.groupby(['decade', 'engine_size_tier'], observed=True)
    .size()
    .unstack(fill_value=0)
    .reindex(columns=engine_labels, fill_value=0)
)
share = counts.div(counts.sum(axis=1), axis=0) * 100
decade_n = counts.sum(axis=1)

fig, ax = plt.subplots(figsize=(12, 6))
share.plot(kind='bar', stacked=True, ax=ax, color=colors, width=0.85)

ax.set_xlabel('Decade')
ax.set_ylabel('Share of dataset (%)')
ax.set_title('Engine size composition by decade')
ax.set_ylim(0, 105)
ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1), title='Engine size')
ax.grid(True, alpha=0.3, axis='y')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print("Share of >2.0L engines per decade :")
print(share[['2.0-3.0L', '> 3.0L']].sum(axis=1).round(1).to_string())
print("\nNumber of cars per decade:")
print(decade_n.to_string())

In [ ]:
mask_15_20 = df["capacity_cm3"].between(1500, 2000, inclusive="both") & df["year_from"].between(1960, 2020) & df["engine_hp"].notna()

motor_15_20 = (
    df.loc[mask_15_20, ["year_from", "engine_hp"]]
    .groupby("year_from")["engine_hp"]
    .agg(
        median="median",
        q25=lambda s: s.quantile(0.25),
        q75=lambda s: s.quantile(0.75),
        n="count",
    )
    .reset_index()
)

motor_15_20 = motor_15_20[motor_15_20["n"] >= 20]

fig, ax = plt.subplots(figsize=(11, 6))
ax.fill_between(
    motor_15_20["year_from"],
    motor_15_20["q25"],
    motor_15_20["q75"],
    alpha=0.2,
    color="steelblue",
    label="IQR",
)
ax.plot(
    motor_15_20["year_from"],
    motor_15_20["median"],
    color="steelblue",
    linewidth=2.2,
    marker="o",
    markersize=3,
    label="Median engine power",
)

ax.set_title("Engine power over time for 1.5–2.0L motors")
ax.set_xlabel("Year")
ax.set_ylabel("Engine power (HP)")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

print(motor_15_20[["year_from", "median", "n"]].tail(10).to_string(index=False))

In [ ]:
#l/100km consumption for 1.5-2.0L motors in city and on highway
mask_15_20_consumption = (
    df["capacity_cm3"].between(1500, 2000, inclusive="both")
    & df["year_from"].between(1960, 2020)
    & df["city_fuel_per_100km_l"].notna()
    & df["highway_fuel_per_100km_l"].notna()
)
consumption_15_20 = (
    df.loc[mask_15_20_consumption, ["year_from", "city_fuel_per_100km_l", "highway_fuel_per_100km_l"]]
    .groupby("year_from")
    .agg(
        city_median=("city_fuel_per_100km_l", "median"),
        city_q25=("city_fuel_per_100km_l", lambda s: s.quantile(0.25)),
        city_q75=("city_fuel_per_100km_l", lambda s: s.quantile(0.75)),
        highway_median=("highway_fuel_per_100km_l", "median"),
        highway_q25=("highway_fuel_per_100km_l", lambda s: s.quantile(0.25)),
        highway_q75=("highway_fuel_per_100km_l", lambda s: s.quantile(0.75)),
        n_city=("city_fuel_per_100km_l", "count"),
        n_highway=("highway_fuel_per_100km_l", "count"),
    )
    .reset_index()
)
consumption_15_20 = consumption_15_20[
    (consumption_15_20["n_city"] >= 20) & (consumption_15_20["n_highway"] >= 20)
]
fig, ax = plt.subplots(figsize=(11, 6))
ax.fill_between(
    consumption_15_20["year_from"],
    consumption_15_20["city_q25"],
    consumption_15_20["city_q75"],
    alpha=0.2,
    color="steelblue",
    label="City IQR",
)
ax.plot(
    consumption_15_20["year_from"],
    consumption_15_20["city_median"],
    color="steelblue",
    linewidth=2.2,
    marker="o",
    markersize=3,
    label="City median",
)
ax.fill_between(
    consumption_15_20["year_from"],
    consumption_15_20["highway_q25"],
    consumption_15_20["highway_q75"],
    alpha=0.2,
    color="crimson",
    label="Highway IQR",
)
ax.plot(
    consumption_15_20["year_from"],
    consumption_15_20["highway_median"],
    color="crimson",
    linewidth=2.2,
    marker="o",
    markersize=3,
    label="Highway median",
)
ax.set_title("Fuel consumption over time for 1.5–2.0L motors")
ax.set_xlabel("Year")
ax.set_ylabel("Fuel consumption (L/100km)")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()